# 01 - Qualidade dos dados

**Objetivo:** entender a estrutura do Steam Games Dataset e diagnosticar problemas de qualidade (Entendimento dos Dados).

**Lê:** `data/original/*.csv` (baixa do Kaggle automaticamente se não existirem)  
**Gera:** `data/clean/dados_limpos.parquet`

In [1]:
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
downloaded = Path("../data/downloaded")
downloaded.mkdir(parents=True, exist_ok=True)
arquivos = ["games_may2024_cleaned.csv"]
# Baixa o dataset só se os CSVs ainda não estiverem em data/downloaded
if not all((downloaded / f).exists() for f in arquivos):
    import kagglehub  # pip install kagglehub
    path = kagglehub.dataset_download("artermiloff/steam-games-dataset")
    for f in arquivos:
        shutil.copy(Path(path) / f, downloaded / f)

print([p.name for p in downloaded.iterdir()])

['games_may2024_cleaned.csv']


In [ ]:

pd.set_option('display.max_columns', None)

dataset = pd.read_csv(downloaded / "games_may2024_cleaned.csv", low_memory=False)

dataset.head()

display(dataset.head())



Duas situações diferentes: valores **NaN** e **zeros que significam "sem dado"** (não aparecem como faltantes, mas são).

In [ ]:
nan = dataset.isna().mean().mul(100).round(1)

zeros_faltantes = pd.Series({"user_score == 0" : (dataset.user_score == 0).mean() * 100,
                             "metacritic_score == 0": (dataset.metacritic_score == 0).mean() * 100,
                             "estimated_owners == 0": (dataset.estimated_owners == "0").mean() * 100,
                             "achievements == 0": (dataset.achievements == 0).mean() * 100,
                             
}).round(1)
print(f"Colunas com NaN declarados: \n{nan[nan > 0 ].sort_values(ascending=False).to_string()}")
print(f"\nZeros que são ausência de dado (%): \n{zeros_faltantes.to_string()}")
print(f"\nAppID duplicado: {dataset.AppID.duplicated().sum()}")

Colunas com NaN declarados: 
score_rank              100.0
metacritic_url           95.3
reviews                  87.7
notes                    83.0
website                  52.5
support_url              49.7
support_email            12.6
detailed_description      0.2
about_the_game            0.2
short_description         0.1

Zeros que são ausência de dado (%): 
user_score == 0          100.0
metacritic_score == 0     95.3
estimated_owners == 0      0.0
achievements == 0         47.2

AppID duplicado: 0


**Decisões:**
- `user_score` é 100% zero: descartada.
- `score_rank` é 100% NaN: descartada támbem.
- `metacritic_score`: 95% zero. Vira NaN e entra só como flag `tem_metacritic`.
- `estimated_owners`, `positive`, `negative`, `recommendations` e `average_playtime_forever` são **consequência** da popularidade (e `estimated_owners` já vem do mesmo processo de coleta). Ficam fora das variáveis preditoras para evitar vazamento; só aparecem no notebook de análise para mostrar o problema.
- `achievements == 0` pode ser "jogo sem conquistas" mesmo, então é mantido como valor real.
- Colunas de texto/URL (descrições, imagens, `notes`, `reviews`) não entram; só viram flags de presença quando fizer sentido.
- `AppID`sem duplicados.
- `release_date` sem datas inválidas. De 1997-06-30 a 2024-05-26 

In [ ]:
dataset = dataset.drop(columns=["user_score"], errors="ignore")
dataset = dataset.drop(columns=["score_rank"], errors="ignore")

dataset["tem_metacritic"] = (dataset["metacritic_score"] > 0).astype(bool)
dataset["metacritic_score"] = dataset["metacritic_score"].replace(0, np.nan)

flag = ["website", "support_url", "support_email", "notes"]
descartar = ["about_the_game", "header_image", "reviews", "screenshots", "movies"]

flag = [c for c in flag if c in dataset.columns]
descartar = [c for c in descartar if c in dataset.columns]

for c in flag:
    dataset[f"tem_{c}"] = dataset[c].notna() & (dataset[c].astype(str).str.strip() !="")
dataset = dataset.drop(columns=flag+descartar)

#print("AppID duplicado:", dataset.AppID.duplicated().sum()) Nenhum duplicado
dataset["release_date"] = pd.to_datetime(dataset["release_date"], errors="coerce")
#print(dataset["release_date"].isna().sum(), "datas inválidas") nenhuma data invalida tb
print(dataset["release_date"].min(), "->", dataset["release_date"].max()) #amplitude das datas de publicacao



1997-06-30 00:00:00 -> 2024-05-26 00:00:00


In [ ]:
clean = Path("../data/clean")
clean.mkdir(parents=True, exist_ok=True)
dataset.to_parquet(clean / "jogos_limpos.parquet", index=False)
print(dataset.shape)

(83646, 40)
